# RSNA Knee Abnormality Detection - inference notebook

Loads every fold checkpoint written by the training notebook, averages logits
across folds and slice groups, converts the mean to per-target percentile ranks
(the metric only reads the order), and writes `submission.csv`.

Offline, single GPU, competition data plus attached weights only. Select
`submission.csv` in the Submit to Competition flow; this notebook does not
submit on its own.

In [ ]:
from pathlib import Path
Path("knee_meta").mkdir(exist_ok=True)

In [ ]:
%%writefile knee_meta/__init__.py
"""RSNA knee abnormality detection: labels, cache, model, training, inference."""
__version__ = "0.1.0"


In [ ]:
%%writefile knee_meta/config.py
"""Meta constants shared by every module and both Kaggle notebooks."""
import os
from pathlib import Path

SEED = 2026

PREPROC = "rgb01"

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
N_LABELS = len(TARGETS)

PLANES = ("Sagittal", "Coronal", "Axial")
SLOTS = [(p, fs) for p in PLANES for fs in (1, 0)]
SLOT_NAMES = ["{}_{}".format(p.upper(), "FLUID" if fs else "STRUCT") for p, fs in SLOTS]
N_SLOT = len(SLOTS)

CROP_MM = 130.0
CACHE_IMG = 336
IMG_SIZES = (336, 224)
TRAIN_IMG_SIZE = 224
EVAL_IMG_SIZE = 224
SLICES_PER_SLOT = 3
SLICE_BAND = (0.20, 0.80)
PLANE_BAND = {
    "Sagittal": (0.12, 0.88),
    "Coronal": (0.20, 0.80),
    "Axial": (0.12, 0.88),
}
LAT_MIN_OFFSET_MM = 20.0

POOL = "cls_mean_focal"
POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5),
    "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4),
    "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5),
    "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5),
    "Effusion": (0, 2),
    "Synovitis": (0, 2),
    "Baker's": (0,),
    "Contusion": (0, 1, 2),
    "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

EPOCHS = 10
BATCH_STUDIES = 8
LR_HEAD = 1e-3
LR_BACKBONE = 8e-6
WEIGHT_DECAY = 0.02
UNFREEZE_LAST = 6
WARMUP_STEPS = 100
PAIRWISE_W = 0.10
GOLD_WEIGHT = 8.0
MISSING_CONF = 0.15
MENTION_CONF = 0.70
POSITIVE_CONF = 0.95

AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.10

N_FOLDS = 5
EVAL_BATCH = 8
TIME_BUDGET_S = 8.0 * 3600
HDR_THREADS = 16
PIX_THREADS = 12

DINOV2_CANDIDATES = (
    "/kaggle/input/models/metaresearch/dinov2/pytorch/small/v1",
    "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1",
    "/kaggle/input/dinov2-small",
)

COMPETITION_SLUG = "rsna-knee-abnormality-detection"
LABEL_FILE = "report_labels.csv"


def competition_roots():
    return [
        Path("/kaggle/input/competitions") / COMPETITION_SLUG,
        Path("/kaggle/input") / COMPETITION_SLUG,
        Path("data"),
        Path("."),
    ]


def find_root():
    for cand in competition_roots():
        if (cand / "test.csv").is_file() and (cand / "test_series").is_dir():
            return cand
    base = Path("/kaggle/input")
    if base.is_dir():
        for d1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for d2 in [d1] + sorted(p for p in d1.iterdir() if p.is_dir()):
                if (d2 / "test.csv").is_file() and (d2 / "test_series").is_dir():
                    return d2
    raise FileNotFoundError("competition mount not found")


def find_backbone():
    for cand in DINOV2_CANDIDATES:
        if Path(cand).is_dir() and (Path(cand) / "config.json").is_file():
            return cand
    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "config.json" in files and "dinov2" in root.lower():
                return root
    return None


In [ ]:
%%writefile knee_meta/labels.py
"""Multilingual rule extraction of the twelve findings from free-text reports.

Every label comes back as (value, confidence). A report that never mentions a
finding is an unknown, not a negative, so it carries a low confidence and pulls
weakly on that output rather than asserting absence.
"""
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from .config import (LABEL_FILE, MENTION_CONF, MISSING_CONF, N_LABELS,
                     POSITIVE_CONF, TARGETS)

_CHAR_MAP = str.maketrans({"ı": "i", "İ": "i", "ß": "ss", "ø": "o", "Ø": "o",
                           "đ": "d", "Đ": "d"})

TEAR = [
    "tear", "torn", "tearing", "rupture", "ruptured", "disruption", "discontinuity",
    "rotura", "ruptura", "desgarro", "roto", "rota",
    "dechirure", "dechire", "lesion meniscale",
    "scheur", "ruptuur", "gescheurd",
    "riss", "rissbildung", "ruptur", "zerreissung", "einriss",
    "yirtik", "yirtig", "kopma", "butunluk kaybi",
    "ρηξη", "ρηξις", "ρηγμα",
    "руптура", "разкъсв", "разрив", "puknuce", "prekid",
]
SPRAIN = [
    "sprain", "esguince", "entorse", "verstauchung", "verstuiking",
    "distorsiyon", "burkulma", "διαστρεμμα", "навяхван",
    "injury", "lesion", "letsel", "verletzung", "zedelenme",
]
OA_FIND = [
    "osteoarthritis", "osteoarthrosis", "arthrosis", "arthritic",
    "artrosis", "artrose", "arthrose", "gonartrose", "gonartroz", "gonarthrose",
    "gonartro", "artroz",
    "chondrosis", "chondral", "chondropathy", "chondropathie", "chondropatie",
    "chondromalacia", "kondromalazi", "condropatia", "condral", "chondropatia",
    "kraakbeenlijden", "kraakbeenverlies", "kraakbeenschade",
    "knorpelschaden", "knorpeldefekt", "knorpelverlust", "knorpelbelag",
    "cartilage loss", "cartilage thinning", "cartilage fissuring",
    "cartilage defect", "cartilage fissure", "chondral defect", "chondral loss",
    "osteophyt", "osteofit", "osteofyt", "osteophyte", "spurring", "osteofyte",
    "joint space narrowing", "gelenkspaltverschmalerung",
    "kikirdak kaybi", "kikirdak incelme", "kondral",
    "ulcera condral", "hrskavice", "denudacija",
    "χονδροπαθ", "οστεοαρθρ", "οστεοφυτ", "αρθριτ",
    "артроз", "хондропат", "остеофит", "хрущялн",
]
MEDIAL_Q = ["medial", "interno", "interna", "interne", "mediaal", "mediale",
            "innen", "medyal", "medijaln", "εσω", "медиал", "вътреш"]
LATERAL_Q = ["lateral", "externo", "externa", "externe", "buiten", "aussen",
             "lateraln", "εξω", "латерал", "външ"]
PF_Q = ["patellofemoral", "patelofemoral", "femoropatellar", "femoropatelar",
        "femoropatellair", "retropatellar", "retrorotulian", "patellar facet",
        "trochlea", "troclea", "trochlee", "rotula", "rotulian", "patella",
        "patellaire", "patellar", "diz kapagi", "patelofemoraln",
        "επιγονατιδ", "τροχιλ", "пател", "ретропател"]
TRICOMP = ["tricompartmental", "tricompartimental", "three compartments",
           "three compartmens", "all compartments", "gonartrose", "gonartroz",
           "gonarthrose", "gonartrosis", "gonartro", "pangonartro"]

SPECIFIC = {
    "ACL": ["acl", "anterior cruciate", "lca", "ligamento cruzado anterior",
            "ligament croise anterieur", "croise anterieur",
            "voorste kruisband", "vkb", "vorderes kreuzband", "vorderen kreuzband",
            "vordere kreuzband", "on capraz bag", "anterior capraz bag",
            "προσθιο χιαστ", "προσθιου χιαστ", "προσθιος χιαστ",
            "предна кръстна", "предната кръстна", "prednji ukrizeni"],
    "MCL": ["mcl", "medial collateral", "lcm", "ligamento colateral medial",
            "ligamento colateral interno", "ligament collateral medial",
            "collateral medial", "mediale collaterale", "mediaal collateraal",
            "innenband", "mediales kollateralband", "medialen kollateralband",
            "mediale kollateralband", "medyal kollateral", "ic yan bag",
            "εσω πλαγιο", "медиален колатерален", "вътрешна колатерална"],
    "Medial Meniscus": [
        "medial meniscus", "meniscus medialis", "menisco medial", "menisco interno",
        "mediale meniscus", "binnenmeniscus", "innenmeniskus", "meniskus medialis",
        "menisque interne", "menisque medial", "medial menisk", "medyal menisk",
        "εσω μηνισκ", "медиалния менискус", "медиален мениск", "вътрешния мениск",
        "medial and lateral menisc", "medial ve lateral menisk",
        "menisco medial y lateral", "menisco interno y externo",
        "menisco interno y lateral", "mediale en laterale meniscus",
        "innen und aussenmeniskus", "medijalnog meniskusa"],
    "Lateral Meniscus": [
        "lateral meniscus", "meniscus lateralis", "menisco lateral",
        "menisco externo", "laterale meniscus", "buitenmeniscus",
        "aussenmeniskus", "meniskus lateralis", "menisque externe",
        "menisque lateral", "lateral menisk", "εξω μηνισκ",
        "латералния менискус", "латерален мениск", "външния мениск",
        "medial and lateral menisc", "medial ve lateral menisk",
        "menisco medial y lateral", "menisco interno y externo",
        "menisco interno y lateral", "mediale en laterale meniscus",
        "innen und aussenmeniskus", "lateralnog meniskusa"],
}
GENERIC = {
    "ACL": ["cruciate", "cruzado", "croise", "kruisband", "kreuzband",
            "χιαστ", "кръстн", "capraz bag", "ukrizen"],
    "MCL": ["collateral", "colateral", "kollateral", "collaterale",
            "πλαγι", "колатерал", "yan bag"],
    "Medial Meniscus": ["menisc", "menisk", "μηνισκ", "мениск"],
    "Lateral Meniscus": ["menisc", "menisk", "μηνισκ", "мениск"],
}
SIDE_Q = {
    "ACL": ["anterior", "anterieur", "voorste", "vorder", "προσθι", "предн", "prednj"],
    "MCL": MEDIAL_Q,
    "Medial Meniscus": MEDIAL_Q,
    "Lateral Meniscus": LATERAL_Q,
}
DIRECT = {
    "Effusion": [
        "effusion", "joint fluid", "hemarthrosis", "haemarthrosis", "hydrops",
        "derrame", "epanchement", "gewrichtsvocht", "vocht in het gewricht",
        "erguss", "gelenkerguss", "gelenkserguss",
        "eklem ici sivi", "sivi artisi", "efuzyon", "eklem mesafesinde sivi",
        "eklem ici serbest sivi", "eklem sivisi",
        "ενδαρθρικ", "αρθρικο υγρο", "συλλογη υγρου",
        "ставен излив", "излив", "хидропс", "zglobni izljev", "izljev"],
    "Synovitis": [
        "synovitis", "synovial thickening", "synovial hypertrophy",
        "thickened synovial", "hypertrophy of the synovium",
        "synovial proliferation", "proliferation of the synovium",
        "sinovitis", "synovite", "synovitiden", "synovialitis",
        "verdikking van het synovium", "verdikkingen van het synovium",
        "synoviale verdikking", "synovialisverdickung", "synovialverdickung",
        "sinovit", "hoffitis", "sinovyal kalinlasma", "sinovyal proliferasyon",
        "συνοβιτ", "υμενιτ", "синовит", "sinovij", "sinovije"],
    "Baker's": [
        "baker", "popliteal cyst", "poplitealcyst", "popliteal cysts",
        "quiste popliteo", "quistes popliteos", "quiste de baker",
        "kyste de baker", "kyste poplite", "popliteale cyste", "popliteale cyst",
        "bakercyste", "bakerzyste", "poplitealzyste", "popliteazyste",
        "baker kisti", "popliteal kist",
        "κυστη baker", "κυστη του baker", "киста на бейкър",
        "бейкърова киста", "poplitealna cista"],
    "Contusion": [
        "contusion", "contusiones", "contusie", "kontusyon", "kontuzyon",
        "bone bruise", "bone bruising", "knochenprellung", "prellung",
        "botcontusie", "μωλωπ", "θλαση", "контузи", "kontuzij", "nagnjecen"],
    "Fracture": [
        "fracture", "fractur", "fractura", "fraktur", "fractuur", "breuk",
        "kirik", "kirig", "καταγμα", "καταγματ", "фрактура", "счупван",
        "avulsion", "avulsie", "avulsiyon", "prijelom"],
}
NEG_SRC = [
    r"\bno\b", r"\bnot\b", r"\bnon\b", r"\bwithout\b", r"\babsen\w*",
    r"\bnegative for\b", r"\bunremarkable\b", r"\bintact\w*", r"\bnormal\w*",
    r"\bpreserved\b", r"\bfree of\b", r"\bexcluded\b",
    r"\bno hay\b", r"\bsin\b", r"\bausen\w*", r"\bno se\b", r"\bconservad\w*",
    r"\bintegr\w*", r"\bindemne\b", r"\bpas de\b", r"\baucun\w*", r"\bsans\b",
    r"\bgeen\b", r"\bzonder\b", r"\bonopvallend\w*", r"\bnormaal\b",
    r"\bvrij van\b", r"\bintacte\b",
    r"\bkein\w*", r"\bohne\b", r"\bunauffallig\w*", r"\bregelrecht\w*",
    r"\bnicht\b", r"\bintakt\w*",
    r"\byok\w*", r"\bizlenmemis\w*", r"\bizlenmedi\w*", r"\bsaptanmamis\w*",
    r"\bgozlenmemis\w*", r"\bgorulmemis\w*", r"\bkorunmus\w*",
    r"\bmevcut degil\b", r"\bnormaldir\b", r"\bdogaldir\b",
    r"\bδεν\b", r"\bχωρις\b", r"\bφυσιολογικ\w*", r"\bακεραι\w*",
    r"\bбез\b", r"\bняма\b", r"\bне се\b", r"\bнормал\w*", r"\bзапазен\w*",
    r"\bсъхранен\w*", r"\bsenza\b", r"\bsem\b", r"\bnao\b", r"\bbez\b",
    r"\buredn\w*",
]

NEG_RE = re.compile("|".join(NEG_SRC))
NEG_BACK = 70
NEG_FWD = 45
QUAL_WINDOW = 90
PAIR_WINDOW = 160

TRAUMA_SRC = [
    r"\bfracture\w*", r"\bfractur\w*", r"\bfraktur\w*", r"\bfractuur\w*",
    r"\bbreuk\w*", r"\bkirik\w*", r"\bavuls\w*", r"\bimpaction\w*",
    r"\bcontus\w*", r"\bcontusion\w*", r"\bbruise\w*", r"\bbruising\b",
    r"\bkontuz\w*", r"\bprellung\w*", r"\bkwetsing\w*", r"\btrauma\w*",
    r"\bakut\w*", r"\bacute\w*", r"\bagud\w*", r"\boedem\w*", r"\bedem\w*",
    r"\bödem\w*", r"\bodem\w*", r"\bedema\w*", r"\bedematoz\b",
    r"\bödematöz\b",
    r"\bκαταγμ\w*", r"\bфрактур\w*", r"\bконтуз\w*", r"\bneglect\w*",
    r"\bsubchondral fracture\w*", r"\bosteochondral injur\w*",
]
TRAUMA_RE = re.compile("|".join(TRAUMA_SRC))
CLAUSE_SPLIT = re.compile(r"[.;:\n\r•·]+|\s-\s|\s>\s|\s\*\s")
PF_MASK = re.compile(
    r"(?:medial|lateral)\s+(?:patellar|patella|facet|faset|trochlea|"
    r"trochlear|retinac)\w*"
    r"|(?:medial|lateral)\w*\s+(?:faset|facet|patel|patell|trochl|retinac)\w*"
    r"|(?:patellar|patella|facet|faset|trochlea|trochlear|retinac|rotul|"
    r"patelo)\w*\s+(?:medial|lateral|superior|inferior)"
    r"|(?:patellofemoral|patelofemoral|femoropatellar|femoropatelar|"
    r"retinacul\w*|retinakul\w*|rotulian\w*|patellar|patella)")

STATUS_NONE = 0
STATUS_NEG = 1
STATUS_POS = 2


def fold_text(text):
    t = str(text).lower().translate(_CHAR_MAP)
    t = unicodedata.normalize("NFKD", t)
    return "".join(c for c in t if not unicodedata.combining(c))


def _find_any(clause, terms):
    best = -1
    for term in terms:
        i = clause.find(term)
        if i >= 0 and (best < 0 or i < best):
            best = i
    return best


def _negated(clause, pos, span):
    back = clause[max(0, pos - NEG_BACK):pos]
    fwd = clause[pos + span:pos + span + NEG_FWD]
    return bool(NEG_RE.search(back) or NEG_RE.search(fwd))


def _direct_status(clause, terms):
    seen = False
    for term in terms:
        start = 0
        while True:
            i = clause.find(term, start)
            if i < 0:
                break
            seen = True
            if not _negated(clause, i, len(term)):
                return STATUS_POS
            start = i + 1
    return STATUS_NEG if seen else STATUS_NONE


def _anatomy_index(clause, label):
    i = _find_any(clause, SPECIFIC[label])
    if i >= 0:
        return i
    g = _find_any(clause, GENERIC[label])
    if g >= 0 and _find_any(clause, SIDE_Q[label]) >= 0:
        return g
    return -1


def _paired_status(clause, label, finds):
    seen = False
    ai = _anatomy_index(clause, label)
    if ai >= 0:
        for term in finds:
            fi = clause.find(term)
            while fi >= 0:
                if abs(fi - ai) <= PAIR_WINDOW:
                    seen = True
                    if not _negated(clause, fi, len(term)):
                        return STATUS_POS
                fi = clause.find(term, fi + 1)
    return STATUS_NEG if seen else STATUS_NONE


def _bump(best, cur):
    return max(best, cur)


def _compartment_status(clause, qualifiers, mask_pf):
    if TRAUMA_RE.search(clause):
        return STATUS_NONE
    best = STATUS_NONE
    seen = False
    for m in re.finditer("|".join(map(re.escape, OA_FIND)), clause):
        oi = m.start()
        if _negated(clause, oi, 8):
            continue
        seen = True
        win = clause[max(0, oi - QUAL_WINDOW):oi + QUAL_WINDOW]
        if mask_pf:
            win = PF_MASK.sub(" ", win)
        if _find_any(win, qualifiers) >= 0:
            best = STATUS_POS
    if _find_any(clause, TRICOMP) >= 0:
        seen = True
        if not _negated(clause, max(0, _find_any(clause, TRICOMP)), 8):
            best = STATUS_POS
    return best if best == STATUS_POS else (STATUS_NEG if seen else STATUS_NONE)


def extract_status(report):
    out = {lab: STATUS_NONE for lab in TARGETS}
    if not isinstance(report, str) or not report.strip():
        return out
    text = fold_text(report)
    for raw in CLAUSE_SPLIT.split(text):
        clause = raw.strip()
        if len(clause) < 3:
            continue
        for lab, terms in DIRECT.items():
            if out[lab] != STATUS_POS:
                out[lab] = _bump(out[lab], _direct_status(clause, terms))
        tear = TEAR + SPRAIN
        for lab in SPECIFIC:
            if out[lab] == STATUS_POS:
                continue
            finds = tear if lab in ("ACL", "MCL") else TEAR
            out[lab] = _bump(out[lab], _paired_status(clause, lab, finds))
        for lab, quals in (("Medial OA", MEDIAL_Q), ("Lateral OA", LATERAL_Q),
                           ("PF OA", PF_Q)):
            if out[lab] != STATUS_POS:
                out[lab] = _bump(out[lab], _compartment_status(
                    clause, quals, mask_pf=lab != "PF OA"))
    return out


def status_to_target(status):
    value = 1.0 if status == STATUS_POS else 0.0
    if status == STATUS_POS:
        conf = POSITIVE_CONF
    elif status == STATUS_NEG:
        conf = MENTION_CONF
    else:
        conf = MISSING_CONF
    return value, conf


def extract_labels(report):
    status = extract_status(report)
    return {lab: status_to_target(status[lab]) for lab in TARGETS}


def build_label_table(train_csv):
    train = pd.read_csv(train_csv)
    cols = list(train.columns)
    rows = []
    for vals in train.itertuples(index=False, name=None):
        rec = dict(zip(cols, vals))
        labels = extract_labels(rec.get("Report"))
        gold = not pd.isna(rec.get("ACL"))
        row = {"StudyInstanceUID": rec["StudyInstanceUID"], "is_gold": int(gold)}
        for lab in TARGETS:
            value = float(rec[lab]) if gold else labels[lab][0]
            row[lab] = value
            row[lab + "__conf"] = 1.0 if gold else labels[lab][1]
        rows.append(row)
    return pd.DataFrame(rows, columns=["StudyInstanceUID", "is_gold"] +
                        [c for lab in TARGETS for c in (lab, lab + "__conf")])


def targets_from_table(table):
    values = table[TARGETS].to_numpy(dtype=np.float32)
    conf = table[[lab + "__conf" for lab in TARGETS]].to_numpy(dtype=np.float32)
    gold = table["is_gold"].to_numpy(dtype=np.float32)
    return values, conf, gold


def save_label_table(train_csv, out_path):
    table = build_label_table(train_csv)
    table.to_csv(out_path, index=False)
    return table


def load_label_table(path):
    return pd.read_csv(path)


CONSENSUS_FILE = "llm_labels_v4_blend.csv"


def resolve_consensus(spec):
    if spec != "auto":
        p = Path(spec)
        if not p.is_file():
            raise FileNotFoundError(f"consensus labels not found: {spec}")
        return p
    roots = [Path("/kaggle/input"), Path.cwd()]
    hits = []
    for root in roots:
        if root.is_dir():
            hits.extend(sorted(root.glob("**/" + CONSENSUS_FILE)))
    if not hits:
        raise FileNotFoundError(
            f"{CONSENSUS_FILE} not found under /kaggle/input or cwd")
    return hits[0]


def apply_consensus(table, partner_path):
    """Downweight cells where a second labeller disagrees with our values.

    Values are untouched (validation truth stays comparable); only the
    training confidence of disagreement cells drops to MISSING_CONF, and
    gold rows are never altered. Returns (table, stats).
    """
    partner = pd.read_csv(partner_path, dtype={"StudyInstanceUID": str})
    idx = partner.set_index("StudyInstanceUID")
    out = table.copy()
    gold = out["is_gold"].to_numpy() == 1
    conf_cols = [lab + "__conf" for lab in TARGETS]
    conf = out[conf_cols].to_numpy(dtype=np.float64).copy()
    before = conf.copy()
    stats = {"partner": Path(partner_path).name, "n_gold_skipped": int(gold.sum()),
             "down": MISSING_CONF, "per_label": {}}
    for j, lab in enumerate(TARGETS):
        if lab not in idx.columns:
            raise KeyError(f"{lab} missing from {partner_path}")
        vals = out["StudyInstanceUID"].map(idx[lab]).astype(float).to_numpy()
        present = ~np.isnan(vals)
        hard = (vals > 0.5) != (out[lab].to_numpy(dtype=float) > 0.5)
        mask = present & hard & ~gold
        conf[mask, j] = MISSING_CONF
        stats["per_label"][lab] = int(mask.sum())
    for j, lab in enumerate(TARGETS):
        out[lab + "__conf"] = conf[:, j]
    stats["n_down"] = int(np.count_nonzero(conf != before))
    return out, stats


def label_columns():
    return TARGETS + [lab + "__conf" for lab in TARGETS] + ["is_gold"]


def audit(table, train_csv):
    train = pd.read_csv(train_csv)
    gold = train.dropna(subset=["ACL"])
    truth = gold[TARGETS].to_numpy(dtype=np.float64) > 0.5
    rule_rows = []
    for rep in gold.Report:
        st = extract_status(rep)
        rule_rows.append([float(st[lab] == STATUS_POS) for lab in TARGETS])
    rule = np.asarray(rule_rows, dtype=np.float64).reshape(-1, len(TARGETS)) > 0.5
    if truth.size == 0:
        return {"n_gold": 0, "macro_agreement": float("nan"),
                "per_label_agreement": {lab: float("nan") for lab in TARGETS},
                "n_labels": N_LABELS, "label_file": LABEL_FILE}
    per_label = dict(zip(TARGETS, (truth == rule).mean(axis=0).tolist()))
    return {
        "n_gold": int(len(gold)),
        "macro_agreement": float((truth == rule).mean()),
        "per_label_agreement": per_label,
        "n_labels": N_LABELS,
        "label_file": LABEL_FILE,
    }


In [ ]:
%%writefile knee_meta/imaging.py
"""DICOM to uint8 cache: physical crop, canonical laterality, slot packing."""
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from .config import (CACHE_IMG, HDR_THREADS, LAT_MIN_OFFSET_MM, N_SLOT,
                     PIX_THREADS, PLANE_BAND, SLICE_BAND,
                     SLICES_PER_SLOT, SLOTS, CROP_MM)

try:
    import pydicom
except ImportError:
    pydicom = None


def _require_pydicom():
    if pydicom is None:
        raise ImportError("pydicom is required to read the DICOM corpus")


_SERIES_INDEX = {}


def _build_index(root, split):
    key = (str(root), split)
    if key in _SERIES_INDEX:
        return _SERIES_INDEX[key]
    idx = {}
    base = Path(root) / split
    if base.is_dir():
        for child in sorted(base.iterdir()):
            if not child.is_dir():
                continue
            idx.setdefault(child.name, child)
            for grand in sorted(child.iterdir()):
                if grand.is_dir():
                    idx.setdefault(grand.name, grand)
    _SERIES_INDEX[key] = idx
    return idx


def find_series_dir(root, uid):
    for split in ("train_series", "test_series"):
        hit = _build_index(root, split).get(uid)
        if hit is not None and hit.exists():
            return hit
    return None


def list_files(series_dir, uid):
    if series_dir is None:
        return []
    if series_dir.is_file():
        return [series_dir]
    files = [p for p in series_dir.rglob("*") if p.is_file()]
    files.sort(key=lambda p: p.name)
    return files


def read_headers(files, threads=HDR_THREADS):
    _require_pydicom()

    def one(path):
        try:
            ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        except Exception:
            return None
        return ds

    with ThreadPoolExecutor(max_workers=threads) as ex:
        out = list(ex.map(one, files))
    return [h for h in out if h is not None]


def _float_seq(value, minimum):
    if value is None:
        return None
    if not isinstance(value, str):
        try:
            out = [float(x) for x in value]
            return out if len(out) >= minimum else None
        except (TypeError, ValueError):
            pass
    text = str(value).strip().strip("[](){}")
    for ch in "\\|,;":
        text = text.replace(ch, " ")
    try:
        out = [float(p) for p in text.split()]
    except ValueError:
        return None
    return out if len(out) >= minimum else None


def _ipp_vec(h, n=3):
    try:
        v = _float_seq(h.ImagePositionPatient, n)
    except Exception:
        return None
    return np.array(v) if v is not None and len(v) >= n else None


def _iop_vec(h, n=6):
    try:
        v = _float_seq(h.ImageOrientationPatient, n)
    except Exception:
        return None
    return np.array(v) if v is not None and len(v) >= n else None


def _pixel_spacing(h):
    try:
        return _float_seq(h.PixelSpacing, 2)
    except Exception:
        return None


def order_slices(headers):
    keys = []
    for i, h in enumerate(headers):
        ipp, iop = _ipp_vec(h), _iop_vec(h)
        if ipp is None or iop is None:
            keys.append((np.inf, i))
            continue
        normal = np.cross(iop[:3], iop[3:6])
        keys.append((float(np.dot(normal, ipp)), i))
    finite = [k for k in keys if np.isfinite(k[0])]
    if len(finite) < max(1, len(keys) // 2):
        return list(range(len(headers)))
    return [i for _, i in sorted(keys, key=lambda t: (t[0], t[1]))]


def laterality(headers):
    cx = []
    for h in headers:
        ipp, iop, ps = _ipp_vec(h), _iop_vec(h), _pixel_spacing(h)
        if ipp is None or iop is None or ps is None:
            continue
        try:
            rows, cols = int(h.Rows), int(h.Columns)
        except Exception:
            continue
        c = ipp + iop[:3] * ps[1] * cols / 2 + iop[3:6] * ps[0] * rows / 2
        cx.append(float(c[0]))
    if not cx:
        return None
    med = float(np.median(cx))
    if abs(med) < LAT_MIN_OFFSET_MM:
        return None
    return "R" if med < 0 else "L"


def crop_resize(arr, ps_rows, ps_cols, out_size, crop_mm=CROP_MM):
    h, w = arr.shape[:2]
    fh, fw = h * ps_rows, w * ps_cols
    if fh >= crop_mm and fw >= crop_mm:
        ch = min(h, int(round(crop_mm / ps_rows)))
        cw = min(w, int(round(crop_mm / ps_cols)))
        y0 = (h - ch) // 2
        x0 = (w - cw) // 2
        arr = arr[y0:y0 + ch, x0:x0 + cw]
        fh, fw = ch * ps_rows, cw * ps_cols
    scale = out_size / max(fh, fw)
    nh, nw = max(1, int(round(fh * scale))), max(1, int(round(fw * scale)))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(arr, (nw, nh), interpolation=interp)
    canvas = np.full((out_size, out_size), int(np.median(resized)), dtype=resized.dtype)
    y0 = (out_size - nh) // 2
    x0 = (out_size - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = resized
    return canvas


def normalise_window(arr, h):
    slope = float(getattr(h, "RescaleSlope", 1.0) or 1.0)
    inter = float(getattr(h, "RescaleIntercept", 0.0) or 0.0)
    return arr.astype(np.float32) * slope + inter


def _window(h):
    wc = getattr(h, "WindowCenter", None)
    ww = getattr(h, "WindowWidth", None)
    try:
        return float(wc), float(ww)
    except (TypeError, ValueError):
        return None, None


def window_to_uint8(arr, centre=None, width=None):
    if centre is None or width is None or not width:
        lo, hi = np.percentile(arr, [1.0, 99.0])
        if hi <= lo:
            lo, hi = float(arr.min()), float(arr.max() + 1.0)
        centre, width = (lo + hi) / 2.0, max(hi - lo, 1.0)
    lo, hi = centre - width / 2.0, centre + width / 2.0
    scaled = np.clip((arr - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    return (scaled * 255.0 + 0.5).astype(np.uint8)


def slot_index(plane, fluid):
    for i, (p, fs) in enumerate(SLOTS):
        if p == plane and fs == int(fluid):
            return i
    return -1


def _sample_indices(n, band, count):
    if n <= 0:
        return []
    if count <= 1:
        return [int(round((band[0] + band[1]) / 2 * (n - 1)))]
    lo = band[0] * (n - 1)
    hi = band[1] * (n - 1)
    step = (hi - lo) / (count - 1) if count > 1 else 0.0
    idx = [int(round(lo + i * step)) for i in range(count)]
    return [min(max(i, 0), n - 1) for i in idx]


def build_cache(root, split, img_size=CACHE_IMG, slices_per_slot=SLICES_PER_SLOT,
                threads=PIX_THREADS, progress=None, max_studies=None):
    _require_pydicom()
    root = str(root)
    series_csv = os.path.join(root, f"{split}_series.csv")
    table = pd.read_csv(series_csv, dtype={"StudyInstanceUID": str,
                                           "SeriesInstanceUID": str})
    studies = sorted(table.StudyInstanceUID.unique())
    if max_studies:
        studies = studies[:max_studies]
    index = {uid: i for i, uid in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, slices_per_slot, img_size, img_size),
                     dtype=np.uint8)
    mask = np.zeros((len(studies), N_SLOT), dtype=np.uint8)

    def decode_one(series_row):
        uid = series_row.StudyInstanceUID
        if uid not in index:
            return None
        sdir = find_series_dir(root, series_row.SeriesInstanceUID)
        files = list_files(sdir, series_row.SeriesInstanceUID)
        if not files:
            return None
        headers = read_headers(files)
        if not headers:
            return None
        order = order_slices(headers)
        headers = [headers[i] for i in order]
        files = [files[i] for i in order]
        plane = str(series_row.Anatomical_Plane)
        fluid = int(series_row.Fluid_Sensitive)
        sidx = slot_index(plane, fluid)
        if sidx < 0:
            return None
        band = PLANE_BAND.get(plane, SLICE_BAND)
        want = _sample_indices(len(headers), band, slices_per_slot + 2)
        if len(headers) > slices_per_slot + 2:
            want = want[1:-1][:slices_per_slot]
        side = laterality(headers)
        slices = []
        for k in want:
            h = headers[k]
            try:
                ds = pydicom.dcmread(str(files[k]), force=True)
                arr = normalise_window(ds.pixel_array, h)
            except Exception:
                arr = None
            if arr is None:
                slices = []
                break
            ps = _pixel_spacing(h)
            if ps is None:
                slices = []
                break
            arr = crop_resize(arr, ps[0], ps[1], img_size)
            wc, ww = _window(h)
            arr = window_to_uint8(arr, wc, ww)
            if side == "L":
                arr = np.ascontiguousarray(arr[:, ::-1])
            slices.append(arr)
        if len(slices) != slices_per_slot:
            return None
        return index[uid], sidx, np.stack(slices)

    rows = [r for r in table.itertuples(index=False)
            if r.StudyInstanceUID in index]
    done = 0
    with ThreadPoolExecutor(max_workers=threads) as ex:
        for result in ex.map(decode_one, rows):
            if result is not None:
                i, sidx, arr = result
                cache[i, sidx] = arr
                mask[i, sidx] = 1
            done += 1
            if progress and done % 200 == 0:
                progress(done, len(rows))
    return cache, mask, np.array(studies, dtype=object)


def save_cache(path, cache, mask, studies):
    np.save(path, cache)
    np.savez(path + ".mask.npz", mask=mask, studies=studies.astype(str))


def load_cache(path):
    cache = np.load(path, allow_pickle=False)
    meta = np.load(path + ".mask.npz", allow_pickle=True)
    return cache, meta["mask"], meta["studies"]


In [ ]:
%%writefile knee_meta/model.py
"""Backbone plus slot-attention head: one bag of slot images to twelve logits."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, ViTConfig, ViTModel

from .config import (N_LABELS, N_SLOT, POOL, POOL_PARTS, SLOT_PRIOR_STRENGTH,
                     SLOT_PRIOR_TABLE, TARGETS, UNFREEZE_LAST)


class SlotHead(nn.Module):
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=True):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == N_SLOT and n_out == N_LABELS:
            for label, slots in SLOT_PRIOR_TABLE.items():
                if label in TARGETS:
                    p_[TARGETS.index(label), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class Model(nn.Module):
    def __init__(self, backbone, dim, pool=POOL, prior=True, img_size=None):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.img_size = img_size
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, N_LABELS, prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def encode(self, x, img_size=None):
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        return torch.cat(parts, dim=1)

    def forward(self, imgs, mask, img_size=None):
        if imgs.dtype == torch.uint8:
            raise ValueError(
                "Model.forward expects float images in [0,1], got uint8; "
                "divide by 255 before calling the model")
        lo, hi = float(imgs.min()), float(imgs.max())
        if hi > 1.5 or lo < -0.5:
            raise ValueError(
                f"Model.forward input range [{lo:.3f}, {hi:.3f}] outside "
                f"[-0.5, 1.5]; expected float images in [0,1]")
        b, s = imgs.shape[:2]
        x = imgs.reshape(b * s, *imgs.shape[2:]).float()
        feat = self.encode(x, img_size if img_size is not None else self.img_size)
        return self.head(feat.reshape(b, s, -1), mask)


def _tiny_backbone(img_size=64):
    cfg = ViTConfig(hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
                    intermediate_size=128, image_size=img_size, patch_size=16,
                    num_channels=3)
    return ViTModel(cfg)


def build_model(unfreeze_last=UNFREEZE_LAST, source=None, pool=POOL, prior=True,
                tiny=False, img_size=None):
    if tiny:
        bb = _tiny_backbone(img_size or 64)
        dim = bb.config.hidden_size
        return Model(bb, dim, pool=pool, prior=prior, img_size=img_size)
    if source is None:
        from .config import find_backbone
        source = find_backbone()
    if source is None:
        raise FileNotFoundError("DINOv2 weights not attached")
    bb = AutoModel.from_pretrained(str(source))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    return Model(bb, bb.config.hidden_size, pool=pool, prior=prior, img_size=img_size)


def param_groups(model, lr_head, lr_backbone, weight_decay):
    head = list(model.head.parameters())
    backbone = [p for p in model.backbone.parameters() if p.requires_grad]
    return [
        {"params": head, "lr": lr_head},
        {"params": backbone, "lr": lr_backbone},
    ], {"weight_decay": weight_decay}


In [ ]:
%%writefile knee_meta/dataset.py
"""Study bags from the uint8 cache, with rigid jitter and no laterality flips."""
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from .config import (AUG_INTENSITY, AUG_ROT_DEG, AUG_SCALE, AUG_SHIFT, N_SLOT,
                     SLICES_PER_SLOT, TARGETS)
from .labels import targets_from_table


def slice_groups(k=SLICES_PER_SLOT):
    return max(1, k // 3)


def pick_group(k, rng, group=None):
    n = slice_groups(k)
    if group is None:
        group = int(rng.integers(n)) if n > 1 else 0
    start = group * 3
    return start, min(start + 3, k)


def augment(x, rng):
    if AUG_ROT_DEG <= 0 and AUG_SCALE <= 0 and AUG_SHIFT <= 0 and AUG_INTENSITY <= 0:
        return x
    h, w = x.shape[-2:]
    angle = float(rng.uniform(-AUG_ROT_DEG, AUG_ROT_DEG)) if AUG_ROT_DEG else 0.0
    scale = float(rng.uniform(1.0 - AUG_SCALE, 1.0 + AUG_SCALE)) if AUG_SCALE else 1.0
    tx = float(rng.uniform(-AUG_SHIFT, AUG_SHIFT)) if AUG_SHIFT else 0.0
    ty = float(rng.uniform(-AUG_SHIFT, AUG_SHIFT)) if AUG_SHIFT else 0.0
    rad = np.deg2rad(angle)
    cos, sin = np.cos(rad) / scale, np.sin(rad) / scale
    theta = torch.tensor([[cos, -sin, tx], [sin, cos, ty]], dtype=x.dtype)
    theta = theta.unsqueeze(0).repeat(x.shape[0], 1, 1)
    grid = F.affine_grid(theta, size=x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border",
                      align_corners=False)
    if AUG_INTENSITY:
        gain = float(rng.uniform(1.0 - AUG_INTENSITY, 1.0 + AUG_INTENSITY))
        bias = float(rng.uniform(-AUG_INTENSITY, AUG_INTENSITY)) * 0.1
        x = x * gain + bias
    return x


class StudyDataset(Dataset):
    def __init__(self, cache, mask, studies, table, train=True, groups=None,
                 img_size=None, study_ids=None):
        self.cache = cache
        self.mask = torch.as_tensor(mask, dtype=torch.float32)
        self.studies = list(studies)
        self.train = train
        self.groups = groups
        self.img_size = img_size
        if study_ids is None:
            order = {s: i for i, s in enumerate(self.studies)}
            table = table[table.StudyInstanceUID.isin(order)]
            rows = table.set_index("StudyInstanceUID").loc[
                [s for s in self.studies if s in set(table.StudyInstanceUID)]]
            values, conf, gold = targets_from_table(rows.reset_index())
            self.uids = list(rows.index)
            self.idx = [order[u] for u in self.uids]
        else:
            pos = {s: i for i, s in enumerate(self.studies)}
            self.uids = [s for s in study_ids if s in pos]
            self.idx = [pos[s] for s in self.uids]
            sub = table.set_index("StudyInstanceUID").loc[self.uids].reset_index()
            values, conf, gold = targets_from_table(sub)
        self.values = torch.as_tensor(values, dtype=torch.float32)
        self.conf = torch.as_tensor(conf, dtype=torch.float32)
        self.gold = torch.as_tensor(gold, dtype=torch.float32)
        self.n_groups = slice_groups(cache.shape[2])

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        row = self.cache[self.idx[i]]
        mask = self.mask[self.idx[i]]
        if self.groups is None:
            group = None
        elif isinstance(self.groups, int):
            group = self.groups
        else:
            group = self.groups[i]
        start, _ = pick_group(row.shape[1], np.random.default_rng(None), group)
        imgs = torch.from_numpy(np.ascontiguousarray(row[:, start:start + 3]))
        if imgs.shape[1] < 3:
            pad = imgs[:, -1:].repeat(1, 3 - imgs.shape[1], 1, 1)
            imgs = torch.cat([imgs, pad], dim=1)
        imgs = imgs.float() / 255.0
        if self.train:
            imgs = augment(imgs, np.random.default_rng(None))
        return {
            "imgs": imgs,
            "mask": mask.clone(),
            "target": self.values[i],
            "conf": self.conf[i],
            "gold": self.gold[i],
        }


def make_dataset(cache, mask, studies, table, train=True, **kw):
    return StudyDataset(cache, mask, studies, table, train=train, **kw)


def make_loader(dataset, batch_size, shuffle, num_workers=0):
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                       shuffle=shuffle, num_workers=num_workers,
                                       drop_last=shuffle and len(dataset) > batch_size)


def empty_batch(batch_size):
    return {
        "imgs": torch.zeros(batch_size, N_SLOT, 3, 0, 0),
        "mask": torch.zeros(batch_size, N_SLOT),
        "target": torch.zeros(batch_size, len(TARGETS)),
        "conf": torch.zeros(batch_size, len(TARGETS)),
        "gold": torch.zeros(batch_size),
    }


In [ ]:
%%writefile knee_meta/metrics.py
"""Macro AUC for the competition metric and per-target percentile ranks."""
import numpy as np
from sklearn.metrics import roc_auc_score

from .config import TARGETS


def macro_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    aucs = []
    for i in range(y_true.shape[1]):
        col = y_true[:, i]
        if col.min() == col.max():
            continue
        aucs.append(roc_auc_score(col, y_score[:, i]))
    return float(np.mean(aucs)) if aucs else float("nan")


def per_label_auc(y_true, y_score):
    out = {}
    for i, lab in enumerate(TARGETS):
        col = np.asarray(y_true)[:, i]
        if col.min() == col.max():
            out[lab] = float("nan")
        else:
            out[lab] = float(roc_auc_score(col, np.asarray(y_score)[:, i]))
    return out


def percentile_rank(scores):
    scores = np.asarray(scores, dtype=np.float64)
    out = np.empty_like(scores)
    n = scores.shape[0]
    order = np.argsort(scores, axis=0, kind="mergesort")
    ranks = np.empty((n, scores.shape[1]), dtype=np.float64)
    for c in range(scores.shape[1]):
        col = order[:, c]
        r = np.empty(n, dtype=np.float64)
        r[col] = np.arange(n, dtype=np.float64)
        vals = scores[col, c]
        i = 0
        while i < n:
            j = i
            while j + 1 < n and vals[j + 1] == vals[i]:
                j += 1
            if j > i:
                r[col[i:j + 1]] = 0.5 * (i + j)
            i = j + 1
        ranks[:, c] = r
    out = (ranks + 0.5) / n
    return out


In [ ]:
%%writefile knee_meta/train.py
"""Five-fold CV training: confidence-weighted BCE plus a pairwise ranking term."""
import argparse
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import KFold

from .config import (BATCH_STUDIES, EPOCHS, EVAL_BATCH, GOLD_WEIGHT, LR_BACKBONE,
                     LR_HEAD, N_FOLDS, PAIRWISE_W, PREPROC, SEED, TIME_BUDGET_S,
                     TRAIN_IMG_SIZE, UNFREEZE_LAST, WEIGHT_DECAY, find_root)
from .dataset import make_dataset, make_loader
from .imaging import load_cache
from .labels import (apply_consensus, audit, load_label_table,
                     resolve_consensus, save_label_table)
from .metrics import macro_auc, per_label_auc
from .model import build_model, param_groups


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def array_stats(arr, target=2_000_000):
    """Mean and std from a strided sample, never a full-array float64 copy."""
    flat = arr.reshape(-1)
    step = max(1, flat.size // target)
    sample = flat[::step]
    return float(sample.mean()), float(sample.std())


def average_states(states):
    """Mean of cloned state dicts; non-float buffers keep the last snapshot."""
    out = {}
    for key in states[0]:
        vals = [s[key] for s in states]
        if torch.is_floating_point(vals[0]):
            out[key] = torch.stack(vals).float().mean(0).to(vals[0].dtype)
        else:
            out[key] = vals[-1].clone()
    return out


def parse_fold_spec(spec):
    """'0' or '0,1,2' -> set of fold ids; None/'' stays None."""
    if spec is None or not str(spec).strip():
        return None
    return {int(x) for x in str(spec).split(",") if str(x).strip()}


def enable_determinism():
    """Strict bitwise-reproducible kernels (CUDA); must run before matmuls."""
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


def determinism_check(model, batch, device, img_size, seed=1234):
    """Two identical forward/backward passes -> (ok, why, strict_unusable)."""
    state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    cpu_rng = torch.get_rng_state()
    cuda_rng = (torch.cuda.get_rng_state_all()
                if torch.cuda.is_available() else None)

    def run_pass():
        model.load_state_dict(state)
        model.zero_grad(set_to_none=True)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        model.train()
        try:
            _, _, _, _, loss = step(model, batch, device, img_size, train=True)
            loss.backward()
        except RuntimeError as exc:
            model.zero_grad(set_to_none=True)
            return None, str(exc)
        grads = {n: p.grad.detach().clone() for n, p in model.named_parameters()
                 if p.grad is not None}
        model.zero_grad(set_to_none=True)
        return grads, None

    g1, e1 = run_pass()
    g2, e2 = run_pass()
    model.load_state_dict(state)
    model.zero_grad(set_to_none=True)
    torch.set_rng_state(cpu_rng)
    if cuda_rng is not None:
        torch.cuda.set_rng_state_all(cuda_rng)
    if e1 is not None or e2 is not None:
        return False, (e1 or e2)[:300], True
    same = all(torch.allclose(g1[k], g2[k], rtol=0.0, atol=0.0, equal_nan=True)
               for k in g1)
    return same, ("bitwise equal" if same else "gradients differ"), False


def weighted_bce(logits, target, conf, gold):
    per = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    if gold.dim() == 1:
        gold = gold.unsqueeze(1)
    w = conf * (1.0 + (GOLD_WEIGHT - 1.0) * gold)
    return (per * w).sum() / w.sum().clamp_min(1e-6)


def pairwise_loss(logits, target, conf, margin=1.0):
    if PAIRWISE_W <= 0:
        return logits.sum() * 0.0
    sign = target[:, None, :] - target[None, :, :]
    valid = sign != 0
    if not bool(valid.any()):
        return logits.sum() * 0.0
    gap = logits[:, None, :] - logits[None, :, :]
    w = torch.minimum(conf[:, None, :], conf[None, :, :])
    loss = F.relu(margin - torch.sign(sign) * gap)
    return (loss * valid.float() * w).sum() / (valid.float() * w).sum().clamp_min(1e-6)


def step(model, batch, device, img_size, train=True):
    imgs = batch["imgs"].to(device, non_blocking=True)
    mask = batch["mask"].to(device, non_blocking=True)
    target = batch["target"].to(device, non_blocking=True)
    conf = batch["conf"].to(device, non_blocking=True)
    gold = batch["gold"].to(device, non_blocking=True)
    logits = model(imgs, mask, img_size=img_size)
    loss = weighted_bce(logits, target, conf, gold)
    if train:
        loss = loss + PAIRWISE_W * pairwise_loss(logits, target, conf)
    return logits.detach(), target, conf, gold, loss


@torch.no_grad()
def evaluate(model, loader, device, img_size):
    model.eval()
    preds, truths = [], []
    total = 0.0
    n = 0
    for batch in loader:
        logits, target, conf, gold, loss = step(model, batch, device, img_size,
                                                train=False)
        preds.append(logits.cpu().numpy())
        truths.append(target.cpu().numpy())
        total += float(loss) * len(target)
        n += len(target)
    preds = np.concatenate(preds)
    truths = np.concatenate(truths)
    return macro_auc(truths, preds), total / max(n, 1), per_label_auc(truths, preds)


def train_fold(fold, tr_idx, va_idx, studies, cache, mask, table, args, device):
    train_ds = make_dataset(cache, mask, studies, table, train=True,
                            study_ids=[studies[i] for i in tr_idx],
                            img_size=args.img_size)
    val_ds = make_dataset(cache, mask, studies, table, train=False,
                          study_ids=[studies[i] for i in va_idx],
                          img_size=args.img_size)
    train_loader = make_loader(train_ds, args.batch_size, shuffle=True)
    val_loader = make_loader(val_ds, args.eval_batch, shuffle=False)
    model = build_model(unfreeze_last=args.unfreeze, source=args.backbone,
                        img_size=args.img_size, tiny=getattr(args, "tiny", False))
    model.to(device)
    groups, extra = param_groups(model, LR_HEAD, LR_BACKBONE, WEIGHT_DECAY)
    opt = torch.optim.AdamW(groups, **extra)
    if getattr(args, "deterministic", False):
        rng_cpu = torch.get_rng_state()
        rng_cuda = (torch.cuda.get_rng_state_all()
                    if torch.cuda.is_available() else None)
        probe = next(iter(train_loader))
        ok, why, strict_unusable = determinism_check(
            model, probe, device, args.img_size)
        print(f"determinism check: {'PASS' if ok else 'FAIL'} ({why})",
              flush=True)
        if strict_unusable:
            torch.use_deterministic_algorithms(False)
            ok, why, _ = determinism_check(model, probe, device, args.img_size)
            print(f"determinism check (best-effort): "
                  f"{'PASS' if ok else 'FAIL'} ({why})", flush=True)
        torch.set_rng_state(rng_cpu)
        if rng_cuda is not None:
            torch.cuda.set_rng_state_all(rng_cuda)
    total_steps = max(1, args.epochs * len(train_loader))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / max(1, args.warmup)) *
        (0.5 * (1.0 + np.cos(np.pi * min(s, total_steps) / total_steps))))
    best, best_state, history = -1.0, None, []
    swa_states = []
    started = time.time()
    for epoch in range(args.epochs):
        model.train()
        total, n = 0.0, 0
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            _, _, _, _, loss = step(model, batch, device, args.img_size, train=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            total += float(loss.detach()) * len(batch["target"])
            n += len(batch["target"])
        val_auc, val_loss, per_label = evaluate(model, val_loader, device,
                                                args.img_size)
        history.append({"epoch": epoch, "train_loss": total / max(n, 1),
                        "val_auc": val_auc, "val_loss": val_loss,
                        "per_label": per_label})
        print(f"fold {fold} epoch {epoch}: train {total / max(n, 1):.4f} "
              f"val_auc {val_auc:.4f}", flush=True)
        if val_auc > best:
            best = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if args.swa_from >= 0 and epoch >= args.swa_from:
            swa_states.append({k: v.detach().cpu().clone()
                               for k, v in model.state_dict().items()})
        if time.time() - started > args.budget_per_fold:
            print(f"fold {fold}: budget reached at epoch {epoch}", flush=True)
            break
    swa = {}
    if swa_states:
        swa_state = average_states(swa_states)
        model.load_state_dict(swa_state)
        swa_auc, swa_loss, _ = evaluate(model, val_loader, device, args.img_size)
        swa = {"swa_val_auc": swa_auc, "swa_members": len(swa_states),
               "swa_from": args.swa_from, "swa_selected": swa_auc > best}
        print(f"fold {fold} swa: {swa_auc:.4f} "
              f"({len(swa_states)} members, selected={swa_auc > best})", flush=True)
        if swa_auc > best:
            best = swa_auc
            best_state = swa_state
    elif args.swa_from >= 0:
        print(f"fold {fold}: no swa members (swa_from={args.swa_from})", flush=True)
    if best_state is not None:
        model.load_state_dict(best_state)
    out = Path(args.out) / f"fold{fold}.pt"
    torch.save({"state_dict": model.state_dict(), "val_auc": best,
                "img_size": args.img_size, "fold": fold, "preproc": PREPROC,
                "history": history, **swa}, out)
    return best, history


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=None)
    ap.add_argument("--cache", default=None)
    ap.add_argument("--labels", default=None)
    ap.add_argument("--out", default="models")
    ap.add_argument("--folds", type=int, default=N_FOLDS)
    ap.add_argument("--epochs", type=int, default=EPOCHS)
    ap.add_argument("--batch-size", type=int, default=BATCH_STUDIES)
    ap.add_argument("--eval-batch", type=int, default=EVAL_BATCH)
    ap.add_argument("--img-size", type=int, default=None)
    ap.add_argument("--unfreeze", type=int, default=UNFREEZE_LAST)
    ap.add_argument("--backbone", default=None)
    ap.add_argument("--warmup", type=int, default=100)
    ap.add_argument("--budget", type=float, default=TIME_BUDGET_S)
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--device", default=None)
    ap.add_argument("--tiny", action="store_true")
    ap.add_argument("--only-fold", type=str, default=None,
                    help="train only these folds, '0' or '0,1,2' (split "
                         "sessions for long runs)")
    ap.add_argument("--swa-from", type=int, default=-1,
                    help="average weights from this epoch onward; -1 disables")
    ap.add_argument("--deterministic", action="store_true",
                    help="reproducible kernels (paired screening)")
    ap.add_argument("--consensus", default=None,
                    help="path or 'auto': downweight cells where a second "
                         "labeller disagrees (values unchanged)")
    args = ap.parse_args(argv)
    args.only_fold = parse_fold_spec(args.only_fold)
    args.budget_per_fold = args.budget / max(1, args.folds)
    if args.deterministic:
        enable_determinism()
    seed_everything(args.seed)

    root = Path(args.root) if args.root else find_root()
    cache_path = Path(args.cache) if args.cache else root / "cache_train.npy"
    label_path = Path(args.labels) if args.labels else Path("report_labels.csv")
    Path(args.out).mkdir(parents=True, exist_ok=True)

    if label_path.is_file():
        table = load_label_table(label_path)
    else:
        table = save_label_table(root / "train.csv", label_path)
    print("label audit:", json.dumps(audit(table, root / "train.csv"), indent=2))
    if args.consensus:
        table, cons_stats = apply_consensus(
            table, resolve_consensus(args.consensus))
        print("consensus:", json.dumps(cons_stats), flush=True)

    cache, mask, studies = load_cache(str(cache_path))
    studies = [str(s) for s in studies]
    known = set(table.StudyInstanceUID)
    keep = [i for i, s in enumerate(studies) if s in known]
    if len(keep) != len(studies):
        cache, mask, studies = cache[keep], mask[keep], [studies[i] for i in keep]
    if args.img_size is None:
        args.img_size = TRAIN_IMG_SIZE
    device = torch.device(args.device or ("cuda" if torch.cuda.is_available()
                                          else "cpu"))
    print(f"cache {tuple(cache.shape)} device {device} img {args.img_size}",
          flush=True)
    print("slot fill:", np.round(mask.mean(axis=0), 3).tolist(),
          "| pixel mean %.1f std %.1f" % array_stats(cache), flush=True)

    order = np.random.RandomState(args.seed).permutation(len(studies))
    splits = KFold(n_splits=args.folds, shuffle=False)
    results = []
    started = time.time()
    for fold, (tr, va) in enumerate(splits.split(order)):
        if args.only_fold is not None and fold not in args.only_fold:
            continue
        auc, history = train_fold(fold, order[tr], order[va], studies, cache,
                                  mask, table, args, device)
        results.append({"fold": fold, "val_auc": auc})
        print(f"fold {fold} best val_auc {auc:.4f}", flush=True)
        if time.time() - started > args.budget:
            print("global budget reached", flush=True)
            break
    summary = Path(args.out) / "cv_summary.json"
    summary.write_text(json.dumps(results, indent=2))
    print("cv:", json.dumps(results), flush=True)
    return results


if __name__ == "__main__":
    main()


In [ ]:
%%writefile knee_meta/infer.py
"""Offline inference: fold and group average, then per-target rank blend."""
import argparse
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from .config import (EVAL_BATCH, EVAL_IMG_SIZE, PREPROC, TARGETS, find_root)
from .imaging import build_cache, load_cache, save_cache
from .metrics import percentile_rank
from .model import build_model

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

MODEL_CANDIDATES = (
    "models",
    "/kaggle/input/rsna-knee-meta-folds-v2",
    "/kaggle/input/datasets/tensorhydra/rsna-knee-meta-folds-v2",
    "/kaggle/working/models",
)


def find_checkpoints(directory):
    p = Path(directory)
    if not p.is_dir():
        return []
    return sorted(p.glob("fold*.pt"))


def resolve_models_dir(preferred=None, candidates=None):
    """Locate fold*.pt checkpoints: explicit dir, then known mount points."""
    env = os.environ.get("RSNA_MODELS_DIR")
    order = [d for d in (env, preferred) if d]
    order += [d for d in (candidates or MODEL_CANDIDATES) if d not in order]
    for d in order:
        if find_checkpoints(d):
            return str(Path(d))
    raise FileNotFoundError(
        "no fold*.pt checkpoints found; looked in: " + ", ".join(order))


def load_models(models_dir, device, backbone=None, unfreeze=None, tiny=False):
    from .config import UNFREEZE_LAST
    paths = sorted(Path(models_dir).glob("fold*.pt"))
    if not paths:
        raise FileNotFoundError(f"no checkpoints in {models_dir}")
    models = []
    for p in paths:
        blob = torch.load(p, map_location="cpu", weights_only=False)
        if blob.get("preproc") != PREPROC:
            raise ValueError(
                f"{p.name}: checkpoint preproc {blob.get('preproc')!r} != "
                f"infer {PREPROC!r}; checkpoint/inference input contract "
                f"mismatch — use matching checkpoints")
        model = build_model(unfreeze_last=unfreeze or UNFREEZE_LAST,
                            source=backbone, img_size=blob.get("img_size"),
                            tiny=tiny)
        model.load_state_dict(blob["state_dict"])
        model.to(device).eval()
        models.append((p.name, model, blob.get("img_size")))
    return models


@torch.no_grad()
def predict(models, cache, mask, device, batch=EVAL_BATCH, groups=None):
    n = cache.shape[0]
    n_groups = max(1, cache.shape[2] // 3)
    groups = list(range(n_groups)) if groups is None else list(groups)
    total = np.zeros((n, len(TARGETS)), dtype=np.float64)
    count = 0
    for _, model, img_size in models:
        for g in groups:
            start = g * 3
            block = cache[:, :, start:start + 3]
            if block.shape[2] < 3:
                pad = np.repeat(block[:, :, -1:], 3 - block.shape[2], axis=2)
                block = np.concatenate([block, pad], axis=2)
            for i in range(0, n, batch):
                x = torch.from_numpy(block[i:i + batch]).float().div(255.0).to(device)
                m = torch.from_numpy(mask[i:i + batch]).float().to(device)
                logits = model(x, m, img_size=img_size)
                total[i:i + batch] += logits.float().cpu().numpy()
            count += 1
    return total / max(count, 1)


def ensure_test_cache(root, img_size, out=None):
    out = Path(out) if out else Path(root) / f"cache_test_{img_size}.npy"
    if out.is_file() and Path(str(out) + ".mask.npz").is_file():
        return load_cache(str(out))
    cache, mask, studies = build_cache(str(root), "test", img_size=img_size,
                                       progress=_progress("test"))
    save_cache(str(out), cache, mask, studies)
    return cache, mask, studies


def _progress(tag):
    if tqdm is None:
        return None
    bar = {"n": 0}

    def hook(done, total):
        if bar["n"] == 0:
            bar["n"] = 1
            bar["bar"] = tqdm(total=total, desc=tag)
        bar["bar"].n = done
        bar["bar"].refresh()
    return hook


def load_template(root):
    """Study order comes from test.csv (guaranteed replaced at scoring).

    sample_submission.csv is only cross-checked: the competition docs list
    test.csv / test_series.csv / test_series/ as the swapped files, so a
    template built from anything else can silently shrink to the 3-row
    example during scoring.
    """
    root = Path(root)
    frames = {}
    for name in ("test.csv", "sample_submission.csv", "submission.csv"):
        p = root / name
        if p.is_file():
            frames[name] = pd.read_csv(p, dtype={"StudyInstanceUID": str})
    if "test.csv" in frames:
        uids = frames["test.csv"].StudyInstanceUID
        source = "test.csv"
    elif "sample_submission.csv" in frames:
        uids = frames["sample_submission.csv"].StudyInstanceUID
        source = "sample_submission.csv"
    elif "submission.csv" in frames:
        uids = frames["submission.csv"].StudyInstanceUID
        source = "submission.csv"
    else:
        raise FileNotFoundError(f"no submission template or test.csv in {root}")
    sample = frames.get("sample_submission.csv")
    if source != "sample_submission.csv" and sample is not None:
        extra = set(uids) - set(sample.StudyInstanceUID)
        dropped = set(sample.StudyInstanceUID) - set(uids)
        if extra or dropped:
            print(f"template cross-check: {source} has {len(uids)} studies, "
                  f"sample_submission has {len(sample)} "
                  f"(+{len(extra)}/-{len(dropped)}); using {source}",
                  flush=True)
    print(f"template: {len(uids)} studies from {source}", flush=True)
    sub = pd.DataFrame({"StudyInstanceUID": list(uids)})
    for lab in TARGETS:
        sub[lab] = 0.5
    return sub


def check_coverage(sub, studies, max_frac=0.02):
    """Template vs cache coverage; raises when most studies would stay 0.5."""
    known = set(studies)
    missing = int(sum(1 for s in sub.StudyInstanceUID if s not in known))
    frac = missing / max(len(sub), 1)
    print(f"coverage: template {len(sub)}, cache {len(studies)}, "
          f"missing {missing} ({frac:.1%})", flush=True)
    if missing and len(sub) > 10 and frac > max_frac:
        raise ValueError(
            f"{missing}/{len(sub)} template studies missing from cache "
            f"({frac:.1%} > {max_frac:.0%}); refusing to submit a "
            f"0.5-filled file")
    if missing and len(sub) <= 10:
        print(f"warning: {missing} template studies missing from cache "
              f"(kept 0.5)", flush=True)
    return missing


def fill_submission(sub, scores, studies, max_frac=0.02):
    """Write scores into the template; small gaps keep 0.5, big gaps raise."""
    if scores.shape != (len(studies), len(TARGETS)):
        raise ValueError(f"scores {scores.shape} does not match "
                         f"{len(studies)} studies x {len(TARGETS)} targets")
    order = {s: i for i, s in enumerate(studies)}
    rows = np.array([order.get(s, -1) for s in sub.StudyInstanceUID])
    known = rows >= 0
    for j, lab in enumerate(TARGETS):
        col = np.full(len(sub), 0.5)
        col[known] = scores[rows[known], j]
        sub[lab] = col
    check_coverage(sub, studies, max_frac=max_frac)
    if len(sub) > 10:
        flat = [lab for lab in TARGETS if (sub[lab] == 0.5).all()]
        if flat:
            raise ValueError(
                f"constant 0.5 column(s) {flat}: predictions carry no "
                f"ranking, the fallback path fired")
    return sub


def check_slots(mask, studies, min_fill=0.5, max_empty_frac=0.02):
    """Decode health: every study enters the cache even when DICOM decode
    fails, so empty slots are the only signal that predictions would be
    constant ties."""
    fill = float(mask.mean()) if mask.size else 0.0
    empty = int((mask.sum(axis=1) == 0).sum()) if mask.size else 0
    n = len(studies)
    print(f"slots: fill {fill:.1%}, studies with no decoded slots "
          f"{empty}/{n}", flush=True)
    if n > 10 and (fill < min_fill or empty > max_empty_frac * n):
        raise ValueError(
            f"slot decode failed: fill {fill:.1%} (< {min_fill:.0%}) or "
            f"{empty}/{n} studies with no slots (> {max_empty_frac:.0%}) — "
            f"predictions would be mostly ties")
    return fill, empty


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=None)
    ap.add_argument("--cache", default=None)
    ap.add_argument("--models", default="models")
    ap.add_argument("--out", default="submission.csv")
    ap.add_argument("--batch", type=int, default=EVAL_BATCH)
    ap.add_argument("--img-size", type=int, default=None)
    ap.add_argument("--backbone", default=None)
    ap.add_argument("--groups", default=None)
    ap.add_argument("--device", default=None)
    ap.add_argument("--tiny", action="store_true")
    ap.add_argument("--rank", action="store_true", default=True)
    args = ap.parse_args(argv)

    root = Path(args.root) if args.root else find_root()
    device = torch.device(args.device or ("cuda" if torch.cuda.is_available()
                                          else "cpu"))
    models_dir = resolve_models_dir(args.models)
    print(f"checkpoints: {models_dir}", flush=True)
    models = load_models(models_dir, device, backbone=args.backbone,
                         tiny=args.tiny)
    img_size = args.img_size or models[0][2] or EVAL_IMG_SIZE

    if args.cache:
        cache, mask, studies = load_cache(args.cache)
    else:
        cache, mask, studies = ensure_test_cache(root, img_size)
    studies = [str(s) for s in studies]
    check_slots(mask, studies)
    sub = load_template(root)
    check_coverage(sub, studies)
    groups = None if args.groups is None else [int(g) for g in
                                                args.groups.split(",")]
    print(f"test cache {tuple(cache.shape)} {len(models)} models", flush=True)
    scores = predict(models, cache, mask, device, batch=args.batch, groups=groups)
    if args.rank:
        scores = percentile_rank(scores)

    fill_submission(sub, scores, studies)
    sub.to_csv(args.out, index=False)
    print(f"wrote {args.out}: {len(sub)} studies, {len(TARGETS)} targets",
          flush=True)
    return sub


if __name__ == "__main__":
    main()


In [ ]:
import json, os, sys, time
from pathlib import Path
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
sys.path.insert(0, os.getcwd())
import torch
from knee_meta.config import (CACHE_IMG, EPOCHS, N_FOLDS, TIME_BUDGET_S,
                              TRAIN_IMG_SIZE, find_root, find_backbone)
root = find_root()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("root    :", root)
print("device  :", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
print("backbone:", find_backbone())
print("cache   :", CACHE_IMG, "px | folds", N_FOLDS, "| epochs", EPOCHS,
      "| train img", TRAIN_IMG_SIZE, "| budget", TIME_BUDGET_S / 3600, "h")
EVAL_IMG = CACHE_IMG

In [ ]:
from knee_meta import imaging
test_cache = Path(f"cache_test_{EVAL_IMG}.npy")
if test_cache.is_file() and Path(str(test_cache) + ".mask.npz").is_file():
    print("test cache present:", test_cache)
else:
    t0 = time.time()
    cache, mask, studies = imaging.build_cache(str(root), "test", img_size=EVAL_IMG)
    imaging.save_cache(str(test_cache), cache, mask, studies)
    print("built", tuple(cache.shape), "in %.1f s" % (time.time() - t0))

In [ ]:
from knee_meta import infer
submission = infer.main([
    "--root", str(root),
    "--cache", f"cache_test_{EVAL_IMG}.npy",
    "--models", "models",
    "--out", "/kaggle/working/submission.csv",
    "--img-size", str(EVAL_IMG),
])
submission.head()